In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn import tree
import numpy as np

#데이터셋 불러오기
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 고윳값 Drop
X_train=train_df.drop(["index","name","health_status"],axis=1)
X_test=test_df.drop(["index","name"],axis=1)
y_train=train_df['health_status']

# 주요 피처 리스트
important_features = ['iron_lvl', 'stature_inch', 'fat_content', 'age_enzyme_ratio', 'Total_Cholesterol']

# 상호작용 피처 생성 함수
def create_interaction_features(df):
    
    # 추가 피처 생성 (주어진 코드대로)
    df['bmi'] = df['mass_kg'] / ((df['stature_inch'] + 100) / 100) ** 2
    df['age'] = 2024 - df['birth'] + 1
    df['bp'] = df['bp_high'] - df['bp_low']
    df['bp_tf'] = (df['bp_high'] > 130) & (df['bp_low'] > 85)
    df['bp_mean'] = (df['bp_high'] + df['bp_low']) / 2
    df['Waist_to_Height_Ratio'] = df['midsection_inch'] / df['stature_inch']
    df['LDL_to_HDL_Ratio'] = df['chol_bad'] / df['chol_good']
    df['Total_Cholesterol'] = df['chol_good'] + df['chol_bad']
    df['Enzyme_Avg'] = df[['enzyme_1', 'enzyme_2', 'enzyme_3']].mean(axis=1)
    df['Vision_Imbalance'] = (df['vision_l'] != df['vision_r'])
    df['vision_mean'] = (df['vision_l'] + df['vision_r']) / 2
    df['Metabolic_Syndrome'] = (
        (df['bp_high'] > 130) |
        (df['bp_low'] > 85) |
        (df['chol_bad'] > 100) |
        (df['midsection_inch'] > 35) |
        (df['sugar_lvl'] > 100)
    )
    df['waist_to_age_ratio'] = df['midsection_inch'] / df['age']
    df['audio_problem'] = ((df['audio_l'] != 'N') | (df['audio_r'] != 'N')).astype(int)
    df['urine_marker_flag'] = (df['urine_marker'] > 0).astype(int)
    df['age_enzyme_ratio'] = df['Enzyme_Avg'] / df['age']
    # 기본적인 상호작용 피처들
    df['iron_fat'] = df['iron_lvl'] * df['fat_content']
    df['stature_age'] = df['stature_inch'] * df['age_enzyme_ratio']
    df['cholesterol_iron'] = df['Total_Cholesterol'] * df['iron_lvl']

    # 나눗셈 상호작용
    df['iron_per_fat'] = df['iron_lvl'] / (df['fat_content'] + 1e-6)
    df['stature_per_cholesterol'] = df['stature_inch'] / (df['Total_Cholesterol'] + 1e-6)
    df['age_per_iron'] = df['age_enzyme_ratio'] / (df['iron_lvl'] + 1e-6)

    # 덧셈 상호작용
    df['iron_plus_fat'] = df['iron_lvl'] + df['fat_content']
    df['age_plus_cholesterol'] = df['age_enzyme_ratio'] + df['Total_Cholesterol']

    # 뺄셈 상호작용
    df['iron_minus_fat'] = df['iron_lvl'] - df['fat_content']
    df['cholesterol_minus_stature'] = df['Total_Cholesterol'] - df['stature_inch']

    # 비선형 상호작용 (제곱, 루트)
    df['iron_squared'] = df['iron_lvl'] ** 2
    df['sqrt_cholesterol'] = df['Total_Cholesterol'] ** 0.5
    
    # 곱셈 기반 상호작용
    df['iron_stature'] = df['iron_lvl'] * df['stature_inch']
    df['age_iron_squared'] = df['age_enzyme_ratio'] * df['iron_squared']
    df['cholesterol_iron'] = df['Total_Cholesterol'] * df['iron_lvl']
    df['bmi_iron'] = df['bmi'] * df['iron_lvl']

    # 나눗셈 기반 상호작용
    df['iron_per_stature'] = df['iron_lvl'] / (df['stature_inch'] + 1e-6)
    df['cholesterol_per_age'] = df['Total_Cholesterol'] / (df['age_enzyme_ratio'] + 1e-6)
    df['stature_per_bmi'] = df['stature_inch'] / (df['bmi'] + 1e-6)

    # 덧셈 및 뺄셈 기반 상호작용
    df['iron_plus_cholesterol'] = df['iron_lvl'] + df['Total_Cholesterol']
    df['stature_minus_mass'] = df['stature_inch'] - df['mass_kg']
    df['cholesterol_minus_age'] = df['Total_Cholesterol'] - df['age_enzyme_ratio']

    # 비선형 상호작용
    df['sqrt_iron_cholesterol'] = np.sqrt(df['iron_lvl'] * df['Total_Cholesterol'])
    df['stature_squared'] = df['stature_inch'] ** 2
    df['log_iron_lvl'] = np.log1p(df['iron_lvl'])
    
    return df

# X_train, X_test 데이터에 적용
X_train = create_interaction_features(X_train)
X_test = create_interaction_features(X_test)

# 결과 확인
X_test


In [ ]:
X_train['audio_r']=X_train['audio_r'].astype('category')
X_train['audio_l']=X_train['audio_l'].astype('category')

X_test['audio_l']=X_test['audio_l'].astype('category')
X_test['audio_r']=X_test['audio_r'].astype('category')


In [ ]:
# 주의: one-hot encoding을 하는 경우 train과 test에 동일하게 적용되어야 함
X = pd.concat([X_train, X_test], axis=0)
X = pd.get_dummies(X)

X_train = X.iloc[:len(X_train)]
X_test = X.iloc[len(X_train):]

{'subsample': 0.95, 'objective': 'binary', 'num_leaves': 45, 'n_estimators': 900, 'max_depth': 6, 'learning_rate': 0.1, 'lambda_l2': 15, 'lambda_l1': 35}

In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import roc_auc_score
import os

# LightGBM 모델 초기화
model = lgb.LGBMClassifier(subsample=0.95, 
                           objective = 'binary', 
                           num_leaves= 45, 
                           n_estimators= 900, 
                           max_depth= 6,
                           learning_rate= 0.1, 
                           lambda_l2= 15, 
                           lambda_l1= 35)

# 피처 임폴턴스 순서대로 피처 드랍 진행
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# 모델 훈련 및 피처 임폴턴스 추출
model.fit(X_train, y_train)
feature_importances = model.feature_importances_
features = X.columns

# 피처 임폴턴스 낮은 순으로 정렬
sorted_features = [feature for _, feature in sorted(zip(feature_importances, features))]

# 결과 저장 파일 정의
filename = 'feature_selection1.0.txt'

# 피처를 하나씩 드랍하면서 KFold로 AUC-ROC 성능 평가
with open(filename, 'a' if os.path.exists(filename) else 'x') as f:
    f.write("Feature Drop Analysis:\n")
    f.write("Initial Features Count: {}\n\n".format(len(features)))
    
    for i in range(len(sorted_features)):
        # 현재 피처 셋 정의
        current_features = sorted_features[i:]
        X_subset = X_train[current_features]

        # KFold 교차 검증
        auc_scores = cross_val_score(model, X_subset, y_train, cv=kfold, scoring='roc_auc')
        mean_auc = np.mean(auc_scores)

        # 결과 파일에 저장
        f.write(f"Dropped Features: {i}, Remaining Features: {len(current_features)}, Mean AUC-ROC: {mean_auc:.4f}\n")
        
    f.write("\n-------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n")


In [ ]:
drop=sorted_features[:19]
sorted_features[:19]

drop=['audio_l_A',
 'audio_l_N',
 'audio_r_N',
 'bp_tf',
 'iron_squared',
 'log_iron_lvl',
 'sqrt_cholesterol',
 'sqrt_iron_cholesterol',
 'stature_squared',
 'urine_marker_flag',
 'Metabolic_Syndrome',
 'Vision_Imbalance',
 'age',
 'audio_r_A',
 'Total_Cholesterol',
 'urine_marker',
 'audio_problem',
 'mass_kg',
 'oral_issues']

In [ ]:
drop=['audio_l_A',
 'audio_l_N',
 'audio_r_N',
 'bp_tf',
 'iron_squared',
 'log_iron_lvl',
 'sqrt_cholesterol',
 'sqrt_iron_cholesterol',
 'stature_squared',
 'urine_marker_flag',
 'Metabolic_Syndrome',
 'Vision_Imbalance',
 'age',
 'audio_r_A',
 'Total_Cholesterol',
 'urine_marker',
 'audio_problem',
 'mass_kg',
 'oral_issues']
X_train=X_train.drop(drop,axis=1)
X_test=X_test.drop(drop,axis=1)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import os
import lightgbm as lgb

model = lgb.LGBMClassifier(random_state=10)

# Define LightGBM parameters for GridSearchCV
params = {
    'lambda_l1': [15, 20, 25, 30, 35],
    'lambda_l2': [15, 20, 25, 30, 35],
    'learning_rate': [0.1, 0.12, 0.15, 0.18, 0.2],
    'max_depth': [5, 6, 7],
    'n_estimators': [600,650,700,750,800],
    'subsample': [0.85, 0.9, 0.95],
    'num_leaves': [40, 45, 50, 55],
}
random_search = RandomizedSearchCV(model, params, n_iter=50,scoring='roc_auc', verbose=3,cv=3)
random_search.fit(X_train, y_train)

# Extract results from the GridSearchCV output
cv_results = random_search.cv_results_

# Write the extracted results to the file

# Define the filename to save the results
filename = 'boost_resultv1.0.txt'
# Write the extracted results to the file (create file if it doesn't exist)
with open(filename, 'a' if os.path.exists(filename) else 'x') as f:
    for mean_score, params in zip(cv_results['mean_test_score'], cv_results['params']):
        f.write(f"Params: {params}, Mean CV Score: {mean_score:.3f}\n")
        
    f.write("\nBest Parameters:\n")
    f.write(f"{random_search.best_params_}\n")
    f.write(f"Best Score: {random_search.best_score_:.3f}\n")
    f.write(f"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n")